# Comparison of models to test explanation of CE signatures

In [10]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [11]:
DATA_LOC = '../data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

# path to save data to on completion
VIS_PATH = os.path.join('data', 'web_vis')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join('data', 'final-document.tsv')

In [12]:
df = pd.read_table(FINAL_DOC, sep='\t')
print(df.shape)
df.head()

(281817, 26)


,x_speaker,x_in_common,x_our_thoughts_synced_up_sr1,x_developed_joint_perspective_sr2,x_thoughts_became_more_alike_sr5,x_saw_world_in_same_way_sr8,y_speaker,y_in_common,y_our_thoughts_synced_up_sr1,y_developed_joint_perspective_sr2,...,turn_delta,nx,ny,Hxy,resid,x_conv_leader,x_responsive_1,x_responsive_2,x_responsive_3,x_you_are_good_listener
0,5e97478f8b670f09dad1fd02,3.0,5.0,5.0,4.0,6.0,5f13bae4a544193e6b94b0cf,5.0,5.0,6.0,...,1,13.0,15.0,2.277085,-0.437443,5.0,5.0,5.0,6.0,7.0
1,5e97478f8b670f09dad1fd02,3.0,5.0,5.0,4.0,6.0,5f13bae4a544193e6b94b0cf,5.0,5.0,6.0,...,3,13.0,16.0,2.241708,-0.460827,5.0,5.0,5.0,6.0,7.0
2,5e97478f8b670f09dad1fd02,3.0,5.0,5.0,4.0,6.0,5f13bae4a544193e6b94b0cf,5.0,5.0,6.0,...,5,13.0,26.0,2.280164,-0.301180,5.0,5.0,5.0,6.0,7.0
3,5e97478f8b670f09dad1fd02,3.0,5.0,5.0,4.0,6.0,5f13bae4a544193e6b94b0cf,5.0,5.0,6.0,...,7,13.0,17.0,1.859337,-0.831346,5.0,5.0,5.0,6.0,7.0
4,5e97478f8b670f09dad1fd02,3.0,5.0,5.0,4.0,6.0,5f13bae4a544193e6b94b0cf,5.0,5.0,6.0,...,9,13.0,6.0,2.359147,-0.465142,5.0,5.0,5.0,6.0,7.0


In [13]:
df['resid2'] = df['resid']**2

In [14]:
df['conversation_id'].nunique()

34

## LME Regression: Predicting CE

In [15]:
import statsmodels.formula.api as smf

In [16]:
test_var = [
    # 'x_conv_leader',
    'x_in_common',
    'x_responsive_1',
    'x_responsive_2',
    'x_responsive_3',
    'x_our_thoughts_synced_up_sr1',
    'x_developed_joint_perspective_sr2',
    'x_thoughts_became_more_alike_sr5',
    'x_saw_world_in_same_way_sr8',
    'x_you_are_good_listener'
]

In [17]:
data = {'params': [], 'aic': []}

In [18]:
for var in tqdm(test_var):
    ######## linear model
    model = "{} ~ resid".format(var)
    md = smf.glm(model,data=df)
    # md = smf.ols(model,data=df)
    mdf = md.fit()
    
    # model parameters table
    result = {'var': var}
    for p in mdf.params.index:
        result[p] = '{} ({})'.format(
            np.format_float_scientific(mdf.params[p], precision=2), 
            np.format_float_scientific(mdf.bse[p], precision=2)
        ) + ''.join(['*'] * (int(mdf.pvalues[p] < .05) + int(mdf.pvalues[p] < .01) + int(mdf.pvalues[p] < .00001) ))
    # result['r2'] = mdf.rsquared
    data['params'] += [result]
    
    # aic table
    aic_result = {
        'var': var, 
        'linear': (2*len(mdf.params)) - (2 * mdf.llf)
    }
    
    ######## square model
    model = "{} ~ resid2".format(var)
    md = smf.glm(model,data=df)
    # md = smf.ols(model,data=df)
    mdf = md.fit()
    
    # model parameters table
    result = {'var': var}
    for p in mdf.params.index:
        result[p] = '{} ({})'.format(
            np.format_float_scientific(mdf.params[p], precision=2), 
            np.format_float_scientific(mdf.bse[p], precision=2)
        ) + ''.join(['*'] * (int(mdf.pvalues[p] < .05) + int(mdf.pvalues[p] < .01) + int(mdf.pvalues[p] < .00001) ))
    # result['r2'] = mdf.rsquared
    data['params'] += [result]
    
    aic_result['squared'] = (2*len(mdf.params)) - (2 * mdf.llf)
    
    data['aic'] += [aic_result]
    
data['params'] = pd.DataFrame(data['params'])
data['aic'] = pd.DataFrame(data['aic'])

  0%|          | 0/9 [00:00<?, ?it/s]

### Looking at models and parameters

In [19]:
data['params'] = data['params'].transpose()
data['params']['var'] = data['params'].index
data['params'] = data['params'][['var'] + [col for col in data['params'].columns if str(col) != 'var']]

In [20]:
data['params'].head()

,var,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
var,var,x_in_common,x_in_common,x_responsive_1,x_responsive_1,x_responsive_2,x_responsive_2,x_responsive_3,x_responsive_3,x_our_thoughts_synced_up_sr1,x_our_thoughts_synced_up_sr1,x_developed_joint_perspective_sr2,x_developed_joint_perspective_sr2,x_thoughts_became_more_alike_sr5,x_thoughts_became_more_alike_sr5,x_saw_world_in_same_way_sr8,x_saw_world_in_same_way_sr8,x_you_are_good_listener,x_you_are_good_listener
Intercept,Intercept,5.60e+00 (3.89e-03)***,5.6e+00 (3.94e-03)***,5.59e+00 (1.87e-03)***,5.59e+00 (1.89e-03)***,5.68e+00 (1.81e-03)***,5.68e+00 (1.83e-03)***,5.94e+00 (1.63e-03)***,5.94e+00 (1.65e-03)***,3.98e+00 (2.50e-03)***,3.98e+00 (2.53e-03)***,5.06e+00 (2.99e-03)***,5.06e+00 (3.03e-03)***,4.58e+00 (3.03e-03)***,4.58e+00 (3.07e-03)***,4.6e+00 (3.09e-03)***,4.6e+00 (3.13e-03)***,7.58e+00 (2.11e-03)***,7.58e+00 (2.14e-03)***
resid,resid,-2.48e-05 (2.78e-03),NaN,-2.35e-05 (1.33e-03),NaN,1.23e-05 (1.29e-03),NaN,-6.42e-06 (1.16e-03),NaN,-7.73e-06 (1.79e-03),NaN,2.94e-05 (2.14e-03),NaN,4.04e-05 (2.16e-03),NaN,-3.59e-05 (2.20e-03),NaN,2.53e-05 (1.51e-03),NaN
resid2,resid2,NaN,3.42e-03 (3.06e-04)***,NaN,-1.17e-04 (1.47e-04),NaN,-8.86e-04 (1.42e-04)***,NaN,-8.06e-04 (1.28e-04)***,NaN,1.72e-04 (1.97e-04),NaN,-1.13e-03 (2.35e-04)***,NaN,-9.75e-04 (2.38e-04)**,NaN,-1.25e-03 (2.43e-04)***,NaN,-1.57e-03 (1.66e-04)***


In [21]:
search_cols = np.array([col for col in np.unique(data['params'].values[0,:]) if col != 'var'])
search_cols

array(['x_developed_joint_perspective_sr2', 'x_in_common',
       'x_our_thoughts_synced_up_sr1', 'x_responsive_1', 'x_responsive_2',
       'x_responsive_3', 'x_saw_world_in_same_way_sr8',
       'x_thoughts_became_more_alike_sr5', 'x_you_are_good_listener'],
      dtype='<U33')

In [22]:
d = []

In [23]:
for col in search_cols:
    sel = (data['params'].values[0,:] == col).nonzero()[0]-1
    vs = data['params'][['var']+sel.tolist()].values
    d += [vs]
d = np.concatenate(d,axis=0)
d = pd.DataFrame(d, columns=['var', '(1)', '(2)'])

In [24]:
sel = d['var'].isin(['var'])
d['var'].loc[sel] = d['(1)'].loc[sel]

In [25]:
data['params']=d

In [26]:
data['params'].head(n=100)

,var,(1),(2)
0,x_developed_joint_perspective_sr2,x_developed_joint_perspective_sr2,x_developed_joint_perspective_sr2
1,Intercept,5.06e+00 (2.99e-03)***,5.06e+00 (3.03e-03)***
2,resid,2.94e-05 (2.14e-03),NaN
3,resid2,NaN,-1.13e-03 (2.35e-04)***
4,x_in_common,x_in_common,x_in_common
5,Intercept,5.60e+00 (3.89e-03)***,5.6e+00 (3.94e-03)***
6,resid,-2.48e-05 (2.78e-03),NaN
7,resid2,NaN,3.42e-03 (3.06e-04)***
8,x_our_thoughts_synced_up_sr1,x_our_thoughts_synced_up_sr1,x_our_thoughts_synced_up_sr1
9,Intercept,3.98e+00 (2.50e-03)***,3.98e+00 (2.53e-03)***


In [27]:
fin_data_path = os.path.join('data', 'parameter-estimates.csv')

In [28]:
data['params'].to_csv(fin_data_path,  header=False, encoding='utf-8')

In [29]:
with open(fin_data_path.replace('.csv', '.txt'), 'w') as f:
    txt =  data['params'].to_latex(index=False).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')

    txt = txt.replace('\\\\', '\\\\\hline').replace('|', '$|$').replace('_delta', ' $\Delta$')
    f.write(txt)
    f.close()

### Comparing AIC scores

In [30]:
aic_data_path = os.path.join('data', 'aic.csv')

In [31]:
data['aic']['model_sel'] = data['aic']['linear'] - data['aic']['squared']
data['aic'].head(n=100)

,var,linear,squared,model_sel
0,x_in_common,1.209016e+06,1.208891e+06,125.129663
1,x_responsive_1,7.949952e+05,7.949946e+05,0.632810
2,x_responsive_2,7.766488e+05,7.766099e+05,38.863013
3,x_responsive_3,7.193039e+05,7.192644e+05,39.457619
4,x_our_thoughts_synced_up_sr1,9.603406e+05,9.603398e+05,0.759247
5,x_developed_joint_perspective_sr2,1.060969e+06,1.060946e+06,23.017460
6,x_thoughts_became_more_alike_sr5,1.068110e+06,1.068094e+06,16.716630
7,x_saw_world_in_same_way_sr8,1.078472e+06,1.078446e+06,26.664262
8,x_you_are_good_listener,8.637957e+05,8.637057e+05,89.951777


In [32]:
data['aic'].to_csv(aic_data_path, index=False, encoding='utf-8')

In [ ]:
with open(aic_data_path.replace('.csv', '.txt'), 'w') as f:
    txt =  data['aic'].to_latex(index=False).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')

    txt = txt.replace('\\\\', '\\\\\hline').replace('|', '$|$').replace('_delta', ' $\Delta$')
    f.write(txt)
    f.close()